In [ ]:
#Importation de toutes les librairies nécessaires
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Téléchargement des mots vides (stop words)
nltk.download('stopwords', quiet=True)

print("Librairies importées avec succès !")

✅ Librairies importées avec succès !


[nltk_data] Error loading stopwords: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1006)>


In [ ]:
#Chargement des données Open Food Facts
import pandas as pd

file_path = '/Users/ambreservaege/Desktop/ML THESE/en.openfoodfacts.org.products.csv'

colonnes_cibles = [
    'countries_tags', 
    'code', 
    'product_name', 
    'brands', 
    'main_category', 
    'categories_tags', 
    'nutriscore_grade', 
    'image_url'
]

try:
    df_raw = pd.read_csv(
        file_path, 
        sep='\t', 
        on_bad_lines='skip',
        usecols=lambda c: c in colonnes_cibles 
    )
    print(f"Succès !")
    print(f"Dimensions : {df_raw.shape[0]} lignes et {df_raw.shape[1]} colonnes.")
    
except Exception as e:
    print(f"Erreur lors du chargement : {e}")

/var/folders/k6/1bq29smd64zb7bjzpz68g0_w0000gn/T/ipykernel_98963/4222635972.py:18: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_raw = pd.read_csv(


✅ Succès !
Dimensions : 4371128 lignes et 8 colonnes.


In [ ]:
#Filtrage sur la France
if 'countries_tags' in df_raw.columns:
    df_france = df_raw[df_raw['countries_tags'].astype(str).str.contains('france', case=False, na=False)].copy()
    print(f"Nombre de produits vendus en France : {df_france.shape[0]}")
else:
    print("La colonne 'countries_tags' est introuvable. On garde toute la base.")
    df_france = df_raw.copy()

print(f"Colonnes conservées pour la suite : {list(df_france.columns)}")

import gc 
del df_raw
gc.collect() 

# 3. Renommage
df = df_france

display(df.head())

Nombre de produits vendus en France : 1248122
🎯 Colonnes conservées pour la suite : ['code', 'product_name', 'brands', 'categories_tags', 'countries_tags', 'nutriscore_grade', 'main_category', 'image_url']


,code,product_name,brands,categories_tags,countries_tags,nutriscore_grade,main_category,image_url
1,3,NaN,NaN,NaN,en:france,NaN,NaN,NaN
2,4,NaN,NaN,NaN,en:france,NaN,NaN,NaN
3,5,NaN,NaN,NaN,en:france,NaN,NaN,NaN
5,6666,6666,NaN,NaN,"en:france,en:germany",unknown,NaN,https://images.openfoodfacts.org/images/produc...
14,15,Madeleines ChocoLait,Apple bandit,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",en:france,e,en:chocolate-madeleines,https://images.openfoodfacts.org/images/produc...


In [3]:
#Traitement des valeurs manquantes et création de la Cible (Y)
# 1. On vérifie le pourcentage de valeurs manquantes
print("--- Pourcentage de valeurs manquantes avant nettoyage ---")
missing = (df.isnull().sum() / len(df)) * 100
print(missing.round(2).astype(str) + ' %')

# 2. Choix de la variable cible (Target)
if 'main_category' in df.columns and df['main_category'].notnull().sum() > 0:
    col_cible = 'main_category'
elif 'categories_tags' in df.columns:
    col_cible = 'categories_tags'
else:
    print("Erreur : Aucune colonne de catégorie trouvée pour l'IA.")
    col_cible = None

if col_cible:
    print(f"\n Colonne choisie pour apprendre les catégories : '{col_cible}'")

    # 3. Suppression stricte des lignes inutilisables (sans nom OU sans catégorie)
    df_clean = df.dropna(subset=['product_name', col_cible]).copy()

    # 4. Nettoyage de la cible : 
    # On enlève les préfixes "en:" et "fr:", et si c'est une liste de tags, on garde le premier
    if col_cible == 'categories_tags':
        df_clean['target'] = df_clean[col_cible].apply(lambda x: str(x).split(',')[0].replace('en:', '').replace('fr:', ''))
    else:
        df_clean['target'] = df_clean[col_cible].astype(str).str.replace('en:', '').str.replace('fr:', '')

    print(f"\n Dimensions après nettoyage : {df_clean.shape[0]} produits 100% exploitables pour l'IA.")

    # Petit aperçu pour vérifier
    display(df_clean[['product_name', 'target']].head())

--- Pourcentage de valeurs manquantes avant nettoyage ---
code                  0.0 %
product_name        10.94 %
brands              45.72 %
categories_tags      49.1 %
countries_tags        0.0 %
nutriscore_grade     2.66 %
main_category        49.1 %
image_url            7.65 %
dtype: object

 Colonne choisie pour apprendre les catégories : 'main_category'

 Dimensions après nettoyage : 619051 produits 100% exploitables pour l'IA.


,product_name,target
14,Madeleines ChocoLait,chocolate-madeleines
19,Nesquik moins de sucre,chocolate-madeleines
24,Frog Fuel Power Protein,protein
26,Volle yoghurt,beverages
27,Hershey’s Syrup,supplement


In [6]:
import nltk
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')
print("Téléchargement terminé !")

Téléchargement terminé !


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ambreservaege/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
#Prétraitement du texte (NLP)
import re
import nltk
from nltk.corpus import stopwords

# Téléchargement du dictionnaire des mots de liaison français (stop words)
nltk.download('stopwords', quiet=True)
stop_words_fr = set(stopwords.words('french'))

def clean_text(text):
    """Fonction qui nettoie le texte pour l'IA"""
    # 1. Tout mettre en minuscules
    text = str(text).lower()
    
    # 2. Remplacer la ponctuation par des espaces
    text = re.sub(r'[^\w\s]', ' ', text)
    
    # 3. Supprimer les chiffres (les poids/volumes perturbent souvent les catégories)
    text = re.sub(r'\d+', '', text)
    
    # 4. Découper en mots (Tokenization)
    mots = text.split()
    
    # 5. Garder les mots pertinents 
    mots_propres = [mot for mot in mots if mot not in stop_words_fr and len(mot) > 1]
    
    # On reforme une phrase propre
    return ' '.join(mots_propres)

# Application de la fonction sur toute la colonne
df_clean['text_cleaned'] = df_clean['product_name'].apply(clean_text)
df_clean = df_clean[df_clean['text_cleaned'].str.strip().astype(bool)]

print(f"\n Nettoyage terminé ! Il reste {df_clean.shape[0]} produits parfaitement propres.")

display(df_clean[['product_name', 'text_cleaned', 'target']].head(10))


 Nettoyage terminé ! Il reste 618824 produits parfaitement propres.


,product_name,text_cleaned,target
14,Madeleines ChocoLait,madeleines chocolait,chocolate-madeleines
19,Nesquik moins de sucre,nesquik moins sucre,chocolate-madeleines
24,Frog Fuel Power Protein,frog fuel power protein,protein
26,Volle yoghurt,volle yoghurt,beverages
27,Hershey’s Syrup,hershey syrup,supplement
29,Lindt Vollmilch Schokolade,lindt vollmilch schokolade,bouillon-cubes
32,Graines de Chia,graines chia,chia
34,Multi Patents Collagen Peptides,multi patents collagen peptides,long-madeleines
36,Madeleines,madeleines,madeleines
41,Butter Burst,butter burst,pt:pao-com-queijo


In [10]:
# Séparation Train / Test (Apprentissage Supervisé)
from sklearn.model_selection import train_test_split

# On ne garde que les catégories avec au moins 50 produits pour un modèle robuste
categories_valides = df_clean['target'].value_counts()[df_clean['target'].value_counts() >= 50].index
df_model = df_clean[df_clean['target'].isin(categories_valides)].copy()

X = df_model['text_cleaned']
y = df_model['target']

# Séparation : 80% pour l'entraînement (Train), 20% pour l'évaluation (Test)
# L'argument stratify=y garantit que les proportions des catégories sont respectées
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Produits pour l'entraînement (Train) : {X_train.shape[0]}")
print(f"Produits pour l'évaluation (Test) : {X_test.shape[0]}")
print(f"Nombre de catégories différentes à prédire : {len(y.unique())}")

Produits pour l'entraînement (Train) : 405884
Produits pour l'évaluation (Test) : 101472
Nombre de catégories différentes à prédire : 2125


In [12]:
#Vectorisation (TF-IDF) et Entraînement ULTRA-RAPIDE (Big Data)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score

vectorizer = TfidfVectorizer(max_features=3000)
X_train_vec = vectorizer.fit_transform(X_train)

X_test_vec = vectorizer.transform(X_test)

# Le SGDClassifier avec loss='log_loss' est la version ultra-rapide de la Régression Logistique
modele = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42, n_jobs=-1)
modele.fit(X_train_vec, y_train)

y_pred = modele.predict(X_test_vec)
accuracy = accuracy_score(y_test, y_pred)

print(f"\n Précision globale (Accuracy) de votre modèle : {accuracy * 100:.2f} %")


 Précision globale (Accuracy) de votre modèle : 30.62 %


In [14]:
#Itération 2 - Optimisation du modèle (Top 30 catégories)
top_30_categories = df_clean['target'].value_counts().head(30).index
print(f"Les 5 catégories principales conservées : {list(top_30_categories[:5])}...")

# 1. On filtre notre base de données pour ne garder que ces catégories
df_top30 = df_clean[df_clean['target'].isin(top_30_categories)].copy()

# 2. On refait notre séparation Train / Test
X_top = df_top30['text_cleaned']
y_top = df_top30['target']
X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(
    X_top, y_top, test_size=0.2, random_state=42, stratify=y_top
)

print(f"Produits conservés pour cette version optimisée : {df_top30.shape[0]}")

# 4. Vectorisation
vectorizer_top = TfidfVectorizer(max_features=3000)
X_train_vec_top = vectorizer_top.fit_transform(X_train_top)
X_test_vec_top = vectorizer_top.transform(X_test_top)

# 5. Nouvel entraînement
modele_optimise = SGDClassifier(loss='log_loss', max_iter=1000, random_state=42, n_jobs=-1)
modele_optimise.fit(X_train_vec_top, y_train_top)

# 6. Nouvelle évaluation
y_pred_top = modele_optimise.predict(X_test_vec_top)
accuracy_top = accuracy_score(y_test_top, y_pred_top)

print(f"\n Nouvelle Précision (Accuracy) : {accuracy_top * 100:.2f} %")

Les 5 catégories principales conservées : ['groceries', 'sweetened-beverages', 'chicken-breasts', 'candies', 'biscuits']...
Produits conservés pour cette version optimisée : 95106

 Nouvelle Précision (Accuracy) : 74.33 %


In [15]:
#Itération 3 N-Grams et Rééquilibrage
# 1. Vectorisation plus intelligente avec ngram_range=(1, 2)
# Cela permet de garder les mots seuls (ex: citron) ET les paires (ex: citron vert)
# On augmente un peu le vocabulaire à 5000 pour faire de la place aux paires de mots
vectorizer_v2 = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))

X_train_vec_v2 = vectorizer_v2.fit_transform(X_train_top)
X_test_vec_v2 = vectorizer_v2.transform(X_test_top)

# 2. Modèle avec class_weight='balanced' pour aider les petites catégories
modele_v2 = SGDClassifier(
    loss='log_loss', 
    max_iter=1000, 
    random_state=42, 
    n_jobs=-1, 
    class_weight='balanced', 
    alpha=0.0001 
)

modele_v2.fit(X_train_vec_v2, y_train_top)

# 3. Évaluation du nouveau modèle
y_pred_v2 = modele_v2.predict(X_test_vec_v2)
accuracy_v2 = accuracy_score(y_test_top, y_pred_v2)

print(f"\n Précision (Accuracy) : {accuracy_v2 * 100:.2f} %")


 Précision (Accuracy) : 78.65 %


In [16]:
#Itération 4 GridSearch et Hyperparamètres
from sklearn.model_selection import GridSearchCV

# 1. On définit la grille des réglages à tester
parametres = {
    'alpha': [0.0001, 0.001, 0.01],
    'penalty': ['l2', 'elasticnet']
}

# 2. On configure le GridSearch avec une validation croisée (cv=3)
grid_search = GridSearchCV(
    estimator=SGDClassifier(loss='log_loss', max_iter=1000, random_state=42, class_weight='balanced'),
    param_grid=parametres,
    cv=3, 
    scoring='accuracy',
    n_jobs=-1 
)

# 3. On lance la recherche sur nos données vectorisées avec N-Grams (X_train_vec_v2)
grid_search.fit(X_train_vec_v2, y_train_top)

print(f"Les meilleurs réglages trouvés par l'ordinateur : {grid_search.best_params_}")

# 4. On évalue le meilleur modèle trouvé sur nos données de Test
meilleur_modele = grid_search.best_estimator_
y_pred_final = meilleur_modele.predict(X_test_vec_v2)
accuracy_finale = accuracy_score(y_test_top, y_pred_final)

print(f"\n Accuracy après GridSearch : {accuracy_finale * 100:.2f} %")

Les meilleurs réglages trouvés par l'ordinateur : {'alpha': 0.0001, 'penalty': 'l2'}

 Accuracy après GridSearch : 78.65 %


In [17]:
#Itération 5 Feature Engineering & SVM
from sklearn.svm import LinearSVC

# 1. FEATURE ENGINEERING 
df_top30['brands_clean'] = df_top30['brands'].fillna('').astype(str).str.lower()

# On crée notre nouvelle variable (X)
X_combo = df_top30['text_cleaned'] + " " + df_top30['brands_clean']
y_combo = df_top30['target']

# 2. On refait la séparation Train/Test avec cette nouvelle donnée
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_combo, y_combo, test_size=0.2, random_state=42, stratify=y_combo
)

# 3. Vectorisation TF-IDF 
vectorizer_final = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_vec_c = vectorizer_final.fit_transform(X_train_c)
X_test_vec_c = vectorizer_final.transform(X_test_c)

# 4. Le nouveau modèle : Support Vector Machine (LinearSVC)
# dual=False est recommandé quand on a plus de lignes que de colonnes
modele_svm = LinearSVC(random_state=42, class_weight='balanced', dual=False, max_iter=2000)
modele_svm.fit(X_train_vec_c, y_train_c)

# 5. Évaluation finale
y_pred_svm = modele_svm.predict(X_test_vec_c)
accuracy_svm = accuracy_score(y_test_c, y_pred_svm)

print(f"\n avec Marque + SVM : {accuracy_svm * 100:.2f} %")


 avec Marque + SVM : 82.90 %


In [ ]:
#Sauvegarde
import joblib
import os

dossier_projet = '/Users/ambreservaege/Desktop/ML THESE/'
chemin_modele = os.path.join(dossier_projet, 'modele_auchan_svm.pkl')
chemin_vectoriseur = os.path.join(dossier_projet, 'vectoriseur_tfidf.pkl')

joblib.dump(modele_svm, chemin_modele)

joblib.dump(vectorizer_final, chemin_vectoriseur)

print(f"Vos fichiers sont sauvegardés ici : \n {dossier_projet}")

Vos fichiers sont sauvegardés ici : 
👉 /Users/ambreservaege/Desktop/ML THESE/


In [21]:
# Création de la base de données pour Streamlit
import os

colonnes_visuelles = ['product_name', 'brands', 'target', 'image_url', 'nutriscore_grade']

df_catalogue = df_top30[colonnes_visuelles].groupby('target').sample(n=1000, random_state=42, replace=True)

chemin_catalogue = '/Users/ambreservaege/Desktop/ML THESE/catalogue_app.csv'
df_catalogue.to_csv(chemin_catalogue, index=False)

print(f"Catalogue exporté avec succès ici : {chemin_catalogue}")

Catalogue exporté avec succès ici : /Users/ambreservaege/Desktop/ML THESE/catalogue_app.csv
